Project Setup Block 

In [1]:
print('Bismillah')

Bismillah


In [2]:
# ==========================================
# 1. PROJECT SETUP & CONFIGURATION
# ==========================================

# Standard library imports
import warnings
import gc # Garbage collector for memory management

# Third-party imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ------------------------------------------
# Environment Configuration
# ------------------------------------------
# Suppress warnings to keep the notebook clean for GitHub and stakeholders
warnings.filterwarnings('ignore')

# ------------------------------------------
# Pandas Display Options
# ------------------------------------------
# Ensure we can see all columns and a reasonable number of rows
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Format floats to 3 decimal places to avoid scientific notation (crucial for financial data)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

# ------------------------------------------
# Visualization Defaults
# ------------------------------------------
# Set a clean, modern, and professional aesthetic for client presentations
sns.set_theme(style="whitegrid", palette="muted")

# Update matplotlib global parameters
plt.rcParams.update({
    'figure.figsize': (12, 6),      # Default wide aspect ratio suitable for time-series
    'axes.titlesize': 16,           # Clear, readable title sizes
    'axes.titleweight': 'bold',
    'axes.labelsize': 14,           # Axis label sizes
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'figure.dpi': 100,              # High resolution for GitHub rendering
    'axes.spines.top': False,       # Remove top/right borders for a cleaner look
    'axes.spines.right': False
})

# Set random seed for reproducibility
np.random.seed(42)

print("✅ Professional Data Science Environment Setup Complete.")

✅ Professional Data Science Environment Setup Complete.


BLOCK 2

In [5]:
# ==========================================
# 2. DATA LOADING & MEMORY OPTIMIZATION (FINAL)
# ==========================================

import os
import time
import pandas as pd
import gc

# Define file paths using relative directories
RAW_DATA_PATH = os.path.join('..', 'data', 'raw', 'online_retail_II.xlsx')
PROCESSED_DATA_PATH = os.path.join('..', 'data', 'processed', 'online_retail.parquet')

def load_data():
    """
    Loads data from Parquet if available (fast), otherwise loads from Excel (slow),
    defensively fixes all mixed data types, and saves a Parquet checkpoint.
    """
    if os.path.exists(PROCESSED_DATA_PATH):
        print("✅ Found optimized Parquet file. Loading...")
        start_time = time.time()
        df = pd.read_parquet(PROCESSED_DATA_PATH)
        print(f"⏱️ Loaded in {time.time() - start_time:.2f} seconds.")
    else:
        print("⚠️ Optimized Parquet not found. Loading from raw Excel (this may take a few minutes)...")
        start_time = time.time()
        
        # Load both sheets
        df_sheet1 = pd.read_excel(RAW_DATA_PATH, sheet_name='Year 2009-2010')
        df_sheet2 = pd.read_excel(RAW_DATA_PATH, sheet_name='Year 2010-2011')
        
        # Concatenate the two years of data vertically
        df = pd.concat([df_sheet1, df_sheet2], ignore_index=True)
        print(f"⏱️ Loaded from Excel in {time.time() - start_time:.2f} seconds.")
        
        print("🔧 Defensively casting all text/ID columns to strings to prevent Parquet crashes...")
        # Force all non-math columns to strings. 
        # (Note: Customer ID has NaNs, which will temporarily become the string "nan". We will clean this in Phase 3).
        text_columns = ['Invoice', 'StockCode', 'Description', 'Customer ID', 'Country']
        for col in text_columns:
            df[col] = df[col].astype(str)
        
        print("💾 Saving optimized Parquet checkpoint for future runs...")
        df.to_parquet(PROCESSED_DATA_PATH, index=False)
        
        # Free up memory
        del df_sheet1, df_sheet2
        gc.collect() 
        
    return df

# Execute the loading function
df_raw = load_data()

# Inspect the true memory footprint
print("\n--- Dataset Memory Profile ---")
df_raw.info(memory_usage='deep')

⚠️ Optimized Parquet not found. Loading from raw Excel (this may take a few minutes)...
⏱️ Loaded from Excel in 145.03 seconds.
🔧 Defensively casting all text/ID columns to strings to prevent Parquet crashes...
💾 Saving optimized Parquet checkpoint for future runs...

--- Dataset Memory Profile ---
<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  str           
 1   StockCode    1067371 non-null  str           
 2   Description  1062989 non-null  str           
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   str           
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(1), int64(1), str(5)
memory usage: 122.9 MB


phase 3


In [6]:
# ==========================================
# 3. INITIAL INSPECTION
# ==========================================

print("--- First 5 Rows ---")
display(df_raw.head())

print("\n--- Last 5 Rows ---")
display(df_raw.tail())

print("\n--- Numerical Summary ---")
# We only describe numeric columns here. We round to 3 decimal places for readability.
display(df_raw.describe().T.round(3))

print("\n--- Date Range ---")
print(f"First Transaction: {df_raw['InvoiceDate'].min()}")
print(f"Last Transaction:  {df_raw['InvoiceDate'].max()}")

--- First 5 Rows ---


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.950,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.750,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.750,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.100,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.250,13085.0,United Kingdom



--- Last 5 Rows ---


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.100,12680.0,France
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.150,12680.0,France
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.150,12680.0,France
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.950,12680.0,France
1067370,581587,POST,POSTAGE,1,2011-12-09 12:50:00,18.000,12680.0,France



--- Numerical Summary ---


,count,mean,min,25%,50%,75%,max,std
Quantity,1067371.000,9.939,-80995.000,1.000,3.000,10.000,80995.000,172.706
InvoiceDate,1067371,2011-01-02 21:13:55.394029,2009-12-01 07:45:00,2010-07-09 09:46:00,2010-12-07 15:28:00,2011-07-22 10:23:00,2011-12-09 12:50:00,NaN
Price,1067371.000,4.649,-53594.360,1.250,2.100,4.150,38970.000,123.553



--- Date Range ---
First Transaction: 2009-12-01 07:45:00
Last Transaction:  2011-12-09 12:50:00
